In [1]:
# Assignment 6 | Tensorflow Data API


%pip -q install tensorflow-datasets --upgrade
%pip install tensorflow
%pip install numpy


# Importing necessary packages
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Model
import numpy as np
import os, pathlib, datetime
import matplotlib.pyplot as plt


# Confirm tensroflow version
print(tf.__version__)
AUTOTUNE = tf.data.AUTOTUNE  # lets tf.ata tune performance automatically
SEED = 13 
tf.keras.utils.set_random_seed(SEED) # ensure reproducible findings


# creates folder structure for saving models and checkpoints
BASE_DIR = pathlib.Path("artifacts")
(BASE_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
(BASE_DIR / "saved_models").mkdir(parents=True, exist_ok=True)




Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
2.20.0


In [2]:

# Simple CNN (similiar to asign_05, keeping the conv body reusable

def make_cifar10_model():
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Conv2D(32, 3, padding='same', activation='relu', name="conv1")(inputs)
    x = layers.Conv2D


In [3]:
#  Cifar 10 - Load data, Train CNN, and save best model

# Load CIFAR-10 (train/validation/test provided by tfds)

# as_supervised=True gives (image, label) pairs only 
ds_all = tfds.load('cifar10', split=['train','test'], as_supervised=True, with_info=False)
ds_train_raw, ds_test_raw = ds_all


In [4]:
# We need to make a separate validation split because CIFAR-10 doesn't ship one

VAL_FRACTION = 0.1
train_count = 50000 #CIFAR-10 has 50k training samples
val_count = int(train_count * VAL_FRACTION)
train_count_eff = train_count - val_count

# Shuffle once for consistent train & validation split
ds_train_raw = ds_train_raw.shuffle(10_000, seed=SEED, reshuffle_each_iteration=False)
ds_val_raw= ds_train_raw.take(val_count)
ds_train_raw= ds_train_raw.skip(val_count)



In [5]:
# Basic Preprocessing
IMG_SIZE = 32
NUM_CLASSES = 10 
BATCH = 128

# Scale images and resize
def preprocess_cifar(image, label):
    image = tf.cast(image, tf.float32) / 255.0 
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    label = tf.cast(label, tf.int32)
    return image, label



In [6]:
# tf.data pipelines
ds_train = (ds_train_raw
            .map(preprocess_cifar, num_parallel_calls=AUTOTUNE)
            .batch(BATCH)
            .prefetch(AUTOTUNE))

ds_val = (ds_val_raw
          .map(preprocess_cifar, num_parallel_calls=AUTOTUNE)
          .batch(BATCH)
          .prefetch(AUTOTUNE))

ds_test = (ds_test_raw
            .map(preprocess_cifar, num_parallel_calls=AUTOTUNE)
            .batch(BATCH)
            .prefetch(AUTOTUNE))

# map: applies preprocessing in parallel
# batch: groups examples
# prefetvh: overlaps GPU with CPU data loading

In [7]:
# Define CNN model

def make_cifar10_model():
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    # First block
    x = layers.Conv2D(32, 3, padding='same', activation='relu', name="conv1")(inputs)
    x = layers.Conv2D(32, 3, activation='relu', name="conv2")(x)
    x = layers.MaxPool2D()(x)

    # Second Block
    x = layers.Conv2D(64, 3, padding='same', activation='relu', name="conv3")(x)
    x = layers.Conv2D(64, 3, activation='relu', name="conv4")(x)

    # Third Block
    x = layers.Conv2D(128, 3, padding='same', activation='relu', name="conv5")(x)
    x = layers.Conv2D(128, 3, activation='relu', name="conv6")(x)
    conv_output = layers.MaxPool2D(name="last_conv_block")(x)
     #Gonna reuse these layers for Catw & dogs

    # Classification head
    x = layers.Flatten()(conv_output)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    return Model(inputs, outputs, name="cifar10_smallcnn")

cifar_model = make_cifar10_model()

In [8]:
# Compile & train with checkpointing

cifar_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy", 
    metrics=['accuracy']
)

ckpt_path = str(BASE_DIR / "checkpoints" / "cifar10_best.keras")

callbacks = [
    keras.callbacks.ModelCheckpoint(ckpt_path, save_best_only=True, monitor ='val_accuracy'),
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_accuracy')
]

history_cifar = cifar_model.fit(
    ds_train, validation_data=ds_val, epochs=30, callbacks=callbacks
)

# ModelCheckpoint saves the best model automatically
# EarlyStopping halts when val accuracy stops improving. 

Epoch 1/30


2025-11-14 11:13:49.341140: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:396] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608


 28/352 ━━━━━━━━━━━━━━━━━━━━ 2:27 454ms/step - accuracy: 0.1232 - loss: 2.2886

KeyboardInterrupt: 

In [ ]:
# Evaluate and save 

test_loss, test_acc = cifar_model.evaluate(ds_test)
print(f"CIFAR-10 test accuracy: {test_acc:.4f}")

# Save entire final model (architecture + weights)
final_save_path = str(BASE_DIR / "saved_models" / "cifar10_smallccnn_final.keras")
cifar_model.save(final_save_path)


In [ ]:
def plot_history(h, title="Training"):
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(h.history['loss'], label='train')
    plt.plot(h.history['val_loss'], label='val')
    plt.title(f"{title} Loss"); plt.legend
    plt.subplot(1, 2, 2)
    plt.plot(h.history['accuracy'], label = 'trian')
    plt.plot(h.history['val_accuracy'], label='val')
    plt.title(f"{title} Accuracy"); plt.legend()
    plt.show

plot_history(history_cifar, "CIFAR-10 small CNN")

In [ ]:
# Transfer Learning: Cats & Dogs
# Load & split dataset
# TFDS Cats vs Dogs dataset has one "train" split (25k images)
# We'll split manually into 90% train/test and 10% validation.

(ds_cvd_train_full, ds_cvd_test_full), info = tfds.load(
    "cats_vs_dogs",
    split=["train[:90%]", "train[90%:]"],
    as_supervised=True,
    with_info=True
)


In [10]:
# Split train to train/val and preprocess

VAL_FRACTION = 0.1
train_est = int(info.splits['train'].num_examples * 0.9)
val_est = int(train_est * VAL_FRACTION)

ds_cvd_train_full = ds_cvd_train_full.shuffle(10_000, seed=SEED, reshuffle_each_iteration=False)
ds_cvd_val_raw   = ds_cvd_train_full.take(val_est)
ds_cvd_train_raw = ds_cvd_train_full.skip(val_est)

IMG_SIZE_CATS = 32 # Make sure this is 32, since batch size is 64
BATCH = 64

def preprocess_cvd(image, label):
    image = tf.image.resize(image, (IMG_SIZE_CATS, IMG_SIZE_CATS))
    image = tf.cast(image, tf.float32) / 255.0
    return image, tf.cast(label, tf.int32)

ds_cvd_train = ds_cvd_train_raw.map(preprocess_cvd).batch(BATCH).prefetch(AUTOTUNE)
ds_cvd_val   = ds_cvd_val_raw.map(preprocess_cvd).batch(BATCH).prefetch(AUTOTUNE)
ds_cvd_test  = ds_cvd_test_full.map(preprocess_cvd).batch(BATCH).prefetch(AUTOTUNE)


NameError: name 'info' is not defined

In [ ]:
# Load your best saved CIFAR-10 model from disk
best_cifar_model = keras.models.load_model(ckpt_path)

# Extract everything up to the last conv block
conv_base = Model(
    inputs=best_cifar_model.input,
    outputs=best_cifar_model.get_layer("last_conv_block").output
)
conv_base.trainable = False    # freeze all conv weights


In [ ]:
# Preprocessing for transfer learning using CIFAR conv base
# Must resize CVD images to 32x32 to match CIFAR input
def preprocess_cvd_32(image, label):
    image = tf.image.resize(image, (32, 32))
    image = tf.cast(image, tf.float32) / 255.0
    label = tf.cast(label, tf.int32)
    return image, label

# Builds 32x32 cats vs dogs datasets for CIFAR conv_base
ds_cvd_train_32 = (ds_cvd_train_raw
                    .map(preprocess_cvd_32, num_parallel_calls=AUTOTUNE)
                    .batch(BATCH)
                    .prefetch(AUTOTUNE))

ds_cvd_val_32 = (ds_cvd_val_raw
                  .map(preprocess_cvd_32, num_parallel_calls=AUTOTUNE)
                  .batch(BATCH)
                  .prefetch(AUTOTUNE))

ds_cvd_test_32 = (ds_cvd_test_full
                   .map(preprocess_cvd_32, num_parallel_calls=AUTOTUNE)
                   .batch(BATCH)
                   .prefetch(AUTOTUNE))


# Extract features one time from frozen conv base
def extract_features(ds, conv_base):
    feats, labels = [], []
    for imgs, lbls in ds:
        f = conv_base(imgs, training=False)        # forward pass (no grad)
        f = tf.reshape(f, [tf.shape(f)[0], -1])    # flatten per sample
        feats.append(f)
        labels.append(lbls)
    return tf.concat(feats, axis=0), tf.concat(labels, axis=0)






Xtr, ytr = extract_features(ds_cvd_train_32, conv_base)
Xva, yva = extract_features(ds_cvd_val_32, conv_base)
Xte, yte = extract_features(ds_cvd_test_32, conv_base)



In [ ]:
# 32x32 datasets for feature extraction using CIFAR conv base
ds_cvd_train_32 = (ds_cvd_train_raw
                    .map(preprocess_cvd_32, num_parallel_calls=AUTOTUNE)
                    .batch(BATCH)
                    .prefetch(AUTOTUNE))

ds_cvd_val_32 = (ds_cvd_val_raw
                  .map(preprocess_cvd_32, num_parallel_calls=AUTOTUNE)
                  .batch(BATCH)
                  .prefetch(AUTOTUNE))

ds_cvd_test_32 = (ds_cvd_test_full
                   .map(preprocess_cvd_32, num_parallel_calls=AUTOTUNE)
                   .batch(BATCH)
                   .prefetch(AUTOTUNE))


In [ ]:
inputs = layers.Input(shape=(Xtr.shape[1],))
x = layers.Dense(256, activation='relu')(inputs)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)
feat_head = Model(inputs, outputs)

feat_head.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

callbacks_feat = [
    keras.callbacks.ModelCheckpoint(BASE_DIR/"checkpoints/catsdogs_feature_extract_best.keras",
                                    save_best_only=True, monitor='val_accuracy'),
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_accuracy')
]

hist_feat = feat_head.fit(
    Xtr, ytr, validation_data=(Xva, yva),
    epochs=30, batch_size=128, callbacks=callbacks_feat
)

plot_history(hist_feat, "Cats vs Dogs (Feature Extraction)")


In [9]:
# Define augmentation for better generalization
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

# Build new model combining augmentation + frozen conv base + classifier
inputs = layers.Input(shape=(IMG_SIZE_CATS, IMG_SIZE_CATS, 3))
x = data_augmentation(inputs)
x = conv_base(x, training=False)
x = layers.Flatten()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)
frozen_model = Model(inputs, outputs)

frozen_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

callbacks_frozen = [
    keras.callbacks.ModelCheckpoint(BASE_DIR/"checkpoints/catsdogs_frozen_best.keras",
                                    save_best_only=True, monitor='val_accuracy'),
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_accuracy')
]

hist_frozen = frozen_model.fit(
    ds_cvd_train, validation_data=ds_cvd_val, epochs=20, callbacks=callbacks_frozen
)

plot_history(hist_frozen, "Cats vs Dogs (Frozen Base + Augmentation)")


NameError: name 'IMG_SIZE_CATS' is not defined

In [41]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

IMG_VGG = 224     # VGG expects 224×224 input

def preprocess_cifar_for_vgg(image, label):
    image = tf.image.resize(image, (IMG_VGG, IMG_VGG))
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)    # apply VGG normalization
    return image, tf.cast(label, tf.int32)

BATCH = 64
ds_train_vgg = ds_train_raw.map(preprocess_cifar_for_vgg).batch(BATCH).prefetch(AUTOTUNE)
ds_val_vgg   = ds_val_raw.map(preprocess_cifar_for_vgg).batch(BATCH).prefetch(AUTOTUNE)
ds_test_vgg  = ds_test_raw.map(preprocess_cifar_for_vgg).batch(BATCH).prefetch(AUTOTUNE)


In [ ]:
augment_vgg = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
    layers.RandomTranslation(0.05, 0.05)
])


In [ ]:
vgg_base = VGG16(include_top=False, weights='imagenet',
                 input_shape=(IMG_VGG, IMG_VGG, 3))
vgg_base.trainable = False    # freeze all pretrained layers

inputs = layers.Input(shape=(IMG_VGG, IMG_VGG, 3))
x = augment_vgg(inputs)
x = vgg_base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)    # replaces Flatten, fewer params
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
vgg_model = Model(inputs, outputs)


In [ ]:
vgg_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_vgg = [
    keras.callbacks.ModelCheckpoint(BASE_DIR/"checkpoints/cifar10_vgg16_best.keras",
                                    save_best_only=True, monitor='val_accuracy'),
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_accuracy')
]

hist_vgg = vgg_model.fit(
    ds_train_vgg, validation_data=ds_val_vgg, epochs=15, callbacks=callbacks_vgg
)

plot_history(hist_vgg, "CIFAR-10 (VGG16 + Augmentation)")


In [ ]:
test_loss_vgg, test_acc_vgg = vgg_model.evaluate(ds_test_vgg)
print(f"VGG16 Test Accuracy: {test_acc_vgg:.4f}")


In [ ]:
# Fine-tune: unfreeze last few layers, retrain at lower LR
vgg_base.trainable = True
for layer in vgg_base.layers[:-4]:
    layer.trainable = False

vgg_model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

vgg_model.fit(ds_train_vgg, validation_data=ds_val_vgg, epochs=5)
